In [2]:
# import os
# # Force CPU if no GPU is detected by Slurm/CUDA
# if "CUDA_VISIBLE_DEVICES" not in os.environ:
#     os.environ["JAX_PLATFORMS"] = "cpu"

In [3]:
import netket as nk

/blue/yujiabin/awwab.azam/NN_Discrete/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You can plot data with JsonLog.data['Energy'][:-30].plot().

In [4]:
print("NetKet version:", nk.__version__)

NetKet version: 3.21.0


In [5]:
L = 4
lattice = nk.graph.Square(L)
N = lattice.n_nodes
print(N)

16


In [6]:
N_e = 5
hi = nk.hilbert.SpinOrbitalFermions(N, s=None, n_fermions=N_e)

In [7]:
from netket.operator.fermion import create as c_dag
from netket.operator.fermion import destroy as c
from netket.operator.fermion import number as num

$\newcommand{\bsl}[1]{\boldsymbol{#1}}$
$$\mathcal{H}=-t\sum_{\langle{\bsl{i}}, \bsl{j}\rangle{}} \left(c_{\bsl{i}}^{\dagger} c_{\bsl{j}} + c_{\bsl{j}}^{\dagger} c_{\bsl{i}}\right) + V\sum_{\langle{\bsl{i}}, \bsl{j}\rangle{}}n_{\bsl{i}}n_{\bsl{j}}$$


In [8]:
t = 1.0
V = 4.0

H = 0.0
for i, j in lattice.edges():
    H -= t * (c_dag(hi, i) @ c(hi, j) + c_dag(hi, j) @ c(hi, i))
    H += V * (num(hi, i) @ num(hi, j))

print(H.max_conn_size)

/blue/yujiabin/awwab.azam/NN_Discrete/.venv/lib/python3.13/site-packages/netket/operator/_fermion2nd/jax.py:557: UserWarning: 
Consider using `netket.experimental.operator.ParticleNumberAndSpinConservingFermioperator2nd` to reduce the number of connected elements and
considerably reduce the computational cost.
You can convert this operator by calling `netket.experimental.operator.ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd`.

  super()._setup(self)


65


In [9]:
from netket.experimental.operator import ParticleNumberConservingFermioperator2nd

H_pnc = ParticleNumberConservingFermioperator2nd.from_fermionoperator2nd(H)
print(H_pnc.max_conn_size)

21


In [10]:
sparse_H = H_pnc.to_sparse()
print(sparse_H.shape)

(4368, 4368)


In [11]:
from scipy.sparse.linalg import eigsh

eigvals, _ = eigsh(sparse_H, k=2, which="SA")
E_gs = eigvals[0]
print(f"Exact GS energy: {eigvals[0]}")

Exact GS energy: -6.8590135543195885


In [12]:
import jax
import jax.numpy as jnp


# Note: This function can also be found inside of netket, in `nk.jax.logdet_cmplx`, but we implement it here
# for pedagogical purposes.
def _logdet_cmplx(A):
    sign, logabsdet = jnp.linalg.slogdet(A)
    return logabsdet.astype(complex) + jnp.log(sign.astype(complex))

In [13]:
from flax import nnx
from jax.nn.initializers import lecun_normal
from typing import Any
from functools import partial

DType = Any
default_kernel_init = lecun_normal()


class LogSlaterDeterminant(nnx.Module):
    hilbert: nk.hilbert.SpinOrbitalFermions

    def __init__(
        self,
        hilbert,
        kernel_init=default_kernel_init,
        param_dtype=float,
        *,
        rngs: nnx.Rngs,
    ):
        self.hilbert = hilbert

        # To generate random numbers we need to extract the key from the `rngs` object.
        key = rngs.params()

        # the N x Nf matrix of the orbitals
        self.M = nnx.Param(
            kernel_init(
                key,
                (
                    self.hilbert.n_orbitals,
                    self.hilbert.n_fermions,
                ),
                param_dtype,
            )
        )

    def __call__(self, n: jax.Array) -> jax.Array:
        # For simplicity, we write a function that operates on a single configuration of size (N,)
        # and we vectorize it using `jnp.vectorize` with the signature='(n)->()' argument, which specifies
        # that the function is defined to operate on arrays of shape (n,) and return scalars.
        @partial(jnp.vectorize, signature="(n)->()")
        def log_sd(n):
            # Find the positions of the occupied orbitals
            R = n.nonzero(size=self.hilbert.n_fermions)[0]

            # Extract from the (N, Nf) matrix the (Nf, Nf) submatrix of M corresponding to the occupied orbitals.
            A = self.M[R]

            return _logdet_cmplx(A)

        return log_sd(n)

In [14]:
# Create the Slater determinant model, using the seed 0
model = LogSlaterDeterminant(hi, rngs=nnx.Rngs(0))

# Define the Metropolis-Hastings sampler
sa = nk.sampler.MetropolisFermionHop(hi, graph=lattice)

In [15]:
vstate = nk.vqs.MCState(sa, model, n_samples=2**12, n_discard_per_chain=16)

In [16]:
vstate.samples.shape

(16, 256, 16)

In [17]:
# Define the optimizer
op = nk.optimizer.Sgd(learning_rate=0.05)

# Create the VMC (Variational Monte Carlo) driver with SR
gs = nk.driver.VMC_SR(H, op, variational_state=vstate, diag_shift=0.05, mode="real")

Automatic SR implementation choice:  QGT


In [18]:
# Construct the logger to visualize the data later on
slater_log = nk.logging.RuntimeLog()

# Run the optimization for 300 iterations
gs.run(n_iter=300, out=slater_log)

100%|██████████| 300/300 [01:51<00:00,  2.70it/s, Energy=-5.000+0.000j ± 0.037 [σ²=4.9e+00, R̂=1.002]]


(RuntimeLog():
  keys = ['acceptance', 'Energy'],)

In [19]:
sd_energy = vstate.expect(H)
error = abs((sd_energy.mean - E_gs) / E_gs)

print(f"Optimized energy : {sd_energy}")
print(f"Relative error   : {error}")

Optimized energy : -5.060+0.000j ± 0.037 [σ²=4.9e+00, R̂=1.004]
Relative error   : 0.26226521903642824


In [21]:
sd_energy.mean

Array(-5.06013286+1.26708394e-17j, dtype=complex128)